# 06 — Vento ERA5 e comparação com o deslocamento

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/06_era5/06_era5.ipynb)

**Objetivo:** associar cada quadro ao vento do ERA5 (espaço e tempo) e comparar a direção do movimento na imagem com o vento.

**Escopo da professora:** seção 9 (registrar produto, variáveis, unidades, nível, resoluções, período, método; conferir convenções) e pergunta 4.

**Lê:** `01_dados/metadados.json` e `05_tracking/trajetoria.csv`.

**Grava:** `era5/*.nc`, `era5_info.json`, `vento.csv`, `vento_vs_deslocamento.csv`.

**GPU:** não


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "06_era5"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## 1. Credenciais do Copernicus CDS
1. Crie conta no [Climate Data Store](https://cds.climate.copernicus.eu), e **aceite a licença** do dataset `ERA5 hourly data on single levels` (botão ao final do formulário de download).
2. Copie sua chave de API (página do perfil) e guarde nos **Secrets** do Colab como `CDS_API_KEY` (ative "Acesso ao notebook").
3. **Nunca** cole a chave no notebook nem commite o arquivo `~/.cdsapirc`. Se o formato da API mudar, confira a página "How to use the CDS API" no site.

In [ ]:
cds_rc = Path.home() / ".cdsapirc"
if not cds_rc.exists():
    try:
        from google.colab import userdata
        cds_rc.write_text("url: https://cds.climate.copernicus.eu/api\nkey: " + userdata.get("CDS_API_KEY") + "\n")
        print("~/.cdsapirc criado a partir do secret CDS_API_KEY")
    except Exception as e:
        print("Configure o secret CDS_API_KEY (ou crie ~/.cdsapirc manualmente):", e)

## 2. Baixar o ERA5 para o local e o período do vídeo
O pedido cobre uma caixa de ±0,5° em torno da câmera e as horas que englobam o vídeo (com 1 h de folga).
O ERA5 é horário e tem grade de ≈ 0,25° (~25 km): o valor representa **condições em escala de grade**, não o microambiente da câmera.
Datas muito recentes podem não estar disponíveis ainda no ERA5 (há atraso de alguns dias).

In [ ]:
import json, numpy as np, pandas as pd, xarray as xr, cdsapi

meta_all = json.loads((stage_dir(cfg, "01_dados") / "metadados.json").read_text(encoding="utf-8"))
SEQ = list(meta_all)[0]              # troque se houver mais de uma sequência
meta = meta_all[SEQ]
lat, lon = float(meta["latitude"]), float(meta["longitude"])

tr = pd.read_csv(stage_dir(cfg, "05_tracking") / "trajetoria.csv")
tr = tr[tr["sequence"] == SEQ].copy()
tr["t_utc"] = pd.to_datetime(tr["t_utc"], utc=True, format="ISO8601")
tmin, tmax = tr["t_utc"].min(), tr["t_utc"].max()
hours = pd.date_range(tmin.floor("h") - pd.Timedelta(hours=1), tmax.ceil("h") + pd.Timedelta(hours=1), freq="h")

area = cfg["era5"]["area"] or [lat + 0.5, lon - 0.5, lat - 0.5, lon + 0.5]     # [N, W, S, E]
request = {
    "product_type": ["reanalysis"],
    "variable": cfg["era5"]["variables"],
    "year": sorted({h.strftime("%Y") for h in hours}),
    "month": sorted({h.strftime("%m") for h in hours}),
    "day": sorted({h.strftime("%d") for h in hours}),
    "time": sorted({h.strftime("%H:00") for h in hours}),
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": area,
}
era_dir = D / "era5"; era_dir.mkdir(exist_ok=True)
target = era_dir / f"era5_{SEQ}.nc"
if not target.exists():
    cdsapi.Client().retrieve(cfg["era5"]["dataset"], request, str(target))
print("ERA5 em", target)

In [ ]:
ds = xr.open_dataset(target)
tname = "valid_time" if "valid_time" in ds.coords else "time"
print(ds)
step = float(abs(np.diff(ds["latitude"].values)).mean()) if ds["latitude"].size > 1 else None

info = {                                       # registro exigido pelo escopo
    "produto": cfg["era5"]["dataset"], "variaveis": {v: ds[v].attrs.get("units") for v in ds.data_vars},
    "nivel": "superfície (10 m para o vento; 2 m para temperatura)",
    "resolucao_espacial_graus": step, "resolucao_temporal": "horária (UTC)",
    "periodo": [str(hours.min()), str(hours.max())], "area_NWSE": area,
    "metodo_extracao": f"{cfg['era5']['method']} (célula/hora mais próxima)" if cfg["era5"]["method"] == "nearest" else "interpolação espacial e temporal",
    "requisicao": request,
}
(D / "era5_info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

## 3. Vento em cada quadro
Convenções do ERA5: `u10` = componente para **leste**, `v10` = para **norte** (m/s). A direção meteorológica é **de onde** o vento sopra (vento de oeste = 270°).
Confira unidades e sinais antes de comparar (os testes de `tests/test_core.py` cobrem essas convenções).

In [ ]:
from plumas import wind as W

lon_q = lon % 360 if float(ds["longitude"].min()) >= 0 and lon < 0 else lon     # ERA5 pode usar 0–360
sel_kwargs = {"latitude": lat, "longitude": lon_q}

def vento_em(t):
    t64 = np.datetime64(t.tz_convert("UTC").tz_localize(None))
    if cfg["era5"]["method"] == "interp":
        p = ds.interp({tname: t64, **sel_kwargs})
    else:
        p = ds.sel({tname: t64, **sel_kwargs}, method="nearest")
    return float(p["u10"]), float(p["v10"]), (float(p["t2m"]) - 273.15 if "t2m" in p else np.nan)

vals = [vento_em(t) for t in tr["t_utc"]]
tr["u10"], tr["v10"], tr["t2m_C"] = zip(*vals)
tr["vento_vel_ms"] = [W.wind_speed(u, v) for u, v in zip(tr["u10"], tr["v10"])]
tr["vento_de_graus"] = [W.wind_from_deg(u, v) for u, v in zip(tr["u10"], tr["v10"])]
tr["vento_para_graus"] = [W.wind_to_deg(u, v) for u, v in zip(tr["u10"], tr["v10"])]
tr[["frame", "t_utc", "u10", "v10", "t2m_C", "vento_vel_ms", "vento_de_graus"]].to_csv(D / "vento.csv", index=False)
tr[["frame", "t_utc", "u10", "v10", "vento_vel_ms", "vento_de_graus"]].head()

## 4. Comparar com o deslocamento na imagem
Só é observável o movimento **lateral** (perpendicular à visada da câmera). Projetamos o vento nessa direção usando o **azimute da câmera**
e comparamos o **sinal** com `dx_px` (direita = positivo). É uma verificação de compatibilidade, com estas ressalvas:

- ignora perspectiva e a componente ao longo da visada;
- o vento do ERA5 é de superfície (10 m) e em escala de grade; a pluma pode estar a outra altura e sob outro vento local;
- o deslocamento do centroide mistura movimento com crescimento/mudança de forma;
- compatibilidade de direções é **evidência de associação possível, não prova de causa**.

In [ ]:
az = float(meta["camera_azimuth_deg"])
tr["vento_lateral_ms"] = [W.lateral_wind_component(u, v, az) for u, v in zip(tr["u10"], tr["v10"])]
tr["concordancia"] = [W.sign_agreement(dx, lat_ms) if pd.notna(dx) else "indefinido"
                      for dx, lat_ms in zip(tr["dx_px"], tr["vento_lateral_ms"])]
cols = ["frame", "t_utc", "track_id", "dx_px", "dy_px", "vento_vel_ms", "vento_de_graus", "vento_lateral_ms", "concordancia"]
tr[cols].to_csv(D / "vento_vs_deslocamento.csv", index=False)

cont = tr["concordancia"].value_counts()
definidos = cont.get("concorda", 0) + cont.get("discorda", 0)
print(cont.to_dict())
if definidos:
    print(f"Entre os passos com sinal definido: {100 * cont.get('concorda', 0) / definidos:.0f}% concordam (n={definidos}).")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ok = tr.dropna(subset=["dx_px"])
ax[0].scatter(ok["vento_lateral_ms"], ok["dx_px"], c=ok["t_utc"].astype("int64"), s=14)
ax[0].axhline(0, c="k", lw=.5); ax[0].axvline(0, c="k", lw=.5)
ax[0].set_xlabel("vento lateral projetado (m/s, + = direita da imagem)"); ax[0].set_ylabel("Δu do centroide (px)")
ax[1].plot(tr["t_utc"], tr["vento_vel_ms"]); ax[1].set_ylabel("vel. do vento ERA5 (m/s)"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(D / "vento_vs_deslocamento.png", dpi=150); plt.show()

In [ ]:
log_experiment(cfg, STAGE, {"era5": info["produto"], "metodo": cfg["era5"]["method"], "sequencia": SEQ,
                            "azimute_camera_deg": az, "concordancia": cont.to_dict(),
                            "decisao": "comparação só de sinal lateral; sem inferência causal"})